# 🦴 02 — Extract Keypoints (Video-Level Split)

> **Thay đổi chính:** Split theo **VIDEO** trước (không phải window) → không còn data leakage
>
> **Pipeline:**
> 1. Mỗi class: chia videos → 70% train / 15% val / 15% test  
> 2. Extract windows từ từng tập riêng biệt  
> 3. Augment chỉ train  
> 4. Normalize (scaler fit trên train)  
> 5. Lưu `.npy`
>
> **Output:** `X_train.npy`, `X_val.npy`, `X_test.npy`, `y_*.npy`, `label_map.json`, `scaler_params.json`

In [1]:
# ==== CELL 1: Cài thư viện ====
import subprocess, sys
pkgs = ['mediapipe>=1.0', 'protobuf>=5.26', 'tqdm', 'opencv-python', 'numpy', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=True)
print('Packages OK')

Packages OK


In [2]:
# ==== CELL 2: Import ====
import os, json, urllib.request
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    import ipywidgets  # noqa: F401
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

print('mediapipe :', mp.__version__)
print('opencv    :', cv2.__version__)
print('numpy     :', np.__version__)

mediapipe : 1.2.3
opencv    : 5.1.0
numpy     : 2.2.6


In [3]:
# ==== CELL 3: Tải model PoseLandmarker ====
MODEL_PATH = 'pose_landmarker_full.task'
MODEL_URL  = (
    'https://storage.googleapis.com/mediapipe-models/'
    'pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task'
)
if not os.path.exists(MODEL_PATH):
    print('Đang tải model (~30MB)...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Tải xong:', MODEL_PATH)
else:
    print('Model đã có:', MODEL_PATH, f'({os.path.getsize(MODEL_PATH)/1024/1024:.1f} MB)')

Model đã có: pose_landmarker_full.task (9.0 MB)


In [4]:
# ==== CELL 4: Config ====
KAGGLE_INPUT = '/kaggle/input/workoutfitness-video'
LOCAL_INPUT  = r'd:\University_DUT\Semester7\Hoc_may_ung_dung\Project_CK\ExercisePoseAssessment\archive'

ARCHIVE_DIR = KAGGLE_INPUT if os.path.exists(KAGGLE_INPUT) else LOCAL_INPUT
OUTPUT_DIR  = '/kaggle/working' if os.path.exists(KAGGLE_INPUT) else \
              r'd:\University_DUT\Semester7\Hoc_may_ung_dung\Project_CK\ExercisePoseAssessment\data\processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Archive dir:', ARCHIVE_DIR)
print('Output dir :', OUTPUT_DIR)

LABEL_MAP = {0:'push-up', 1:'lateral raise', 2:'squat', 3:'pull Up', 4:'leg raises'}
VIDEO_EXT  = ('.mp4', '.avi', '.mov', '.MP4', '.AVI', '.MOV')

TARGET_FPS    = 30
WINDOW_FRAMES = 30
STEP_FRAMES   = 5
N_FEATURES    = 33 * 3   # 99

# VIDEO-LEVEL split ratio
TEST_RATIO    = 0.15     # 15% video làm test
VAL_RATIO     = 0.15     # 15% video làm val (trong số còn lại)
RANDOM_SEED   = 42

print(f'Config: window={WINDOW_FRAMES}f, step={STEP_FRAMES}, features={N_FEATURES}')
print(f'Video split: train~70% / val~15% / test~15%')

Archive dir: d:\University_DUT\Semester7\Hoc_may_ung_dung\Project_CK\ExercisePoseAssessment\archive
Output dir : d:\University_DUT\Semester7\Hoc_may_ung_dung\Project_CK\ExercisePoseAssessment\data\processed
Config: window=30f, step=5, features=99
Video split: train~70% / val~15% / test~15%


In [5]:
# ==== CELL 5: Init MediaPipe ====
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)
DETECTOR = mp_vision.PoseLandmarker.create_from_options(options)
print('MediaPipe PoseLandmarker (Tasks API) initialized')

MediaPipe PoseLandmarker (Tasks API) initialized


In [6]:
# ==== CELL 6: Helper Functions ====

def extract_keypoints(frame_rgb: np.ndarray):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result   = DETECTOR.detect(mp_image)
    if not result.pose_landmarks:
        return None
    lms = result.pose_landmarks[0]
    return np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()


def resample_frames(cap: cv2.VideoCapture, target_fps: int = 30):
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if src_fps <= 0 or total <= 0:
        return []
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(f)
    if abs(src_fps - target_fps) < 0.5:
        return frames
    duration = len(frames) / src_fps
    n_target = max(1, int(round(duration * target_fps)))
    indices  = np.linspace(0, len(frames)-1, n_target, dtype=int)
    return [frames[i] for i in indices]


def forward_fill(kp_list: list):
    result = []
    last   = np.zeros(N_FEATURES, dtype=np.float32)
    for kp in kp_list:
        if kp is not None: last = kp
        result.append(last.copy())
    return result


def sliding_window(kp_list: list, window=30, step=5):
    arr  = np.array(kp_list, dtype=np.float32)
    seqs = []
    if len(arr) < window: return seqs
    for s in range(0, len(arr)-window+1, step):
        seqs.append(arr[s:s+window])
    return seqs


def process_video(path: str):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print(f'  WARN: cannot open {path}')
        return []
    frames = resample_frames(cap, TARGET_FPS)
    cap.release()
    kps = [extract_keypoints(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
    return forward_fill(kps)


print('Helper functions OK')

Helper functions OK


In [7]:
# ==== CELL 7: Augmentation Functions ====

def aug_flip(seq):        # Lật ngang
    s = seq.copy(); s[:, 0::3] = 1.0 - s[:, 0::3]; return s

def aug_slow(seq, f=2):   # Giảm tốc độ
    sub = seq[::f]
    if len(sub) < WINDOW_FRAMES:
        pad = np.tile(sub[-1:], (WINDOW_FRAMES-len(sub), 1))
        sub = np.vstack([sub, pad])
    return sub[:WINDOW_FRAMES].astype(np.float32)

def aug_noise(seq, s=0.005):  # Gaussian noise
    return (seq + np.random.normal(0, s, seq.shape)).astype(np.float32)


def augment_train(X, y):
    """4× augmentation: gốc + flip + slow + flip+noise."""
    Xs, ys = [X], [y]
    X_flip = np.array([aug_flip(s) for s in X], dtype=np.float32)
    Xs.append(X_flip);                    ys.append(y)
    Xs.append(np.array([aug_slow(s) for s in X], dtype=np.float32)); ys.append(y)
    Xs.append(np.array([aug_noise(s) for s in X_flip], dtype=np.float32)); ys.append(y)
    X_out = np.concatenate(Xs); y_out = np.concatenate(ys)
    print(f'  Augment: {len(X):,} → {len(X_out):,} ({len(X_out)//len(X)}×)')
    return X_out, y_out


print('Augmentation functions OK')

Augmentation functions OK


In [8]:
# ==== CELL 8: VIDEO-LEVEL SPLIT + EXTRACTION ====
# Key change: split VIDEOS trước → extract windows từng tập riêng

train_seqs, train_labels = [], []
val_seqs,   val_labels   = [], []
test_seqs,  test_labels  = [], []

split_stats = {}

for label_id, exercise in LABEL_MAP.items():
    folder = os.path.join(ARCHIVE_DIR, exercise)
    if not os.path.exists(folder):
        print(f'SKIP: {folder} not found'); continue

    all_videos = sorted([f for f in os.listdir(folder) if f.endswith(VIDEO_EXT)])
    n_total    = len(all_videos)

    # ── BƯỚC 1: Chia VIDEOS (không phải windows) ──
    # Tách test trước
    train_val_vids, test_vids = train_test_split(
        all_videos, test_size=TEST_RATIO, random_state=RANDOM_SEED
    )
    # Tách val từ train+val
    # val_ratio tính trong tập train_val để đạt ~15% tổng
    val_ratio_adj = VAL_RATIO / (1 - TEST_RATIO)   # ~0.176
    train_vids, val_vids = train_test_split(
        train_val_vids, test_size=val_ratio_adj, random_state=RANDOM_SEED
    )

    split_stats[exercise] = {
        'total': n_total,
        'train_vids': len(train_vids),
        'val_vids'  : len(val_vids),
        'test_vids' : len(test_vids),
    }
    print(f'\n[{label_id}] {exercise}: {n_total} videos → '
          f'train={len(train_vids)} / val={len(val_vids)} / test={len(test_vids)}')

    # ── BƯỚC 2: Extract windows từ MỖI TẬP RIÊNG ──
    for split_name, vid_list, seq_list, lbl_list in [
        ('train', train_vids, train_seqs, train_labels),
        ('val',   val_vids,   val_seqs,   val_labels),
        ('test',  test_vids,  test_seqs,  test_labels),
    ]:
        n_win = 0
        for vf in tqdm(vid_list, desc=f'  {split_name}', leave=False):
            path   = os.path.join(folder, vf)
            kp_seq = process_video(path)
            for w in sliding_window(kp_seq, WINDOW_FRAMES, STEP_FRAMES):
                seq_list.append(w.astype(np.float32))
                lbl_list.append(label_id)
                n_win += 1
        split_stats[exercise][f'{split_name}_windows'] = n_win
        print(f'    {split_name}: {len(vid_list)} videos → {n_win} windows')

print('\n=== EXTRACTION DONE ===')
print(f'Train windows: {len(train_seqs):,}')
print(f'Val   windows: {len(val_seqs):,}')
print(f'Test  windows: {len(test_seqs):,}')


[0] push-up: 56 videos → train=38 / val=9 / test=9


  train:   0%|          | 0/38 [00:00<?, ?it/s]

    train: 38 videos → 1064 windows


  val:   0%|          | 0/9 [00:00<?, ?it/s]

    val: 9 videos → 345 windows


  test:   0%|          | 0/9 [00:00<?, ?it/s]

... (21 outputs left out)

  val:   0%|          | 0/3 [00:00<?, ?it/s]

    val: 3 videos → 216 windows


  test:   0%|          | 0/4 [00:00<?, ?it/s]

    test: 4 videos → 214 windows

=== EXTRACTION DONE ===
Train windows: 4,806
Val   windows: 1,247
Test  windows: 1,061


In [9]:
# ==== CELL 9: Chuyển sang numpy arrays ====
import collections

X_train_raw = np.array(train_seqs, dtype=np.float32)  # (N, 30, 99)
X_val_raw   = np.array(val_seqs,   dtype=np.float32)
X_test_raw  = np.array(test_seqs,  dtype=np.float32)
y_train_raw = np.array(train_labels, dtype=np.int32)
y_val_raw   = np.array(val_labels,   dtype=np.int32)
y_test_raw  = np.array(test_labels,  dtype=np.int32)

print('=== CLASS DISTRIBUTION (trước augment) ===')
for split_name, y_s in [('Train', y_train_raw), ('Val', y_val_raw), ('Test', y_test_raw)]:
    dist = collections.Counter(y_s.tolist())
    total = sum(dist.values())
    line  = '  '.join(f"{LABEL_MAP[k]}:{dist[k]}" for k in sorted(dist))
    print(f'  {split_name:5} ({total:>5} windows): {line}')

=== CLASS DISTRIBUTION (trước augment) ===
  Train ( 4806 windows): push-up:1064  lateral raise:1188  squat:1163  pull Up:796  leg raises:595
  Val   ( 1247 windows): push-up:345  lateral raise:289  squat:224  pull Up:173  leg raises:216
  Test  ( 1061 windows): push-up:289  lateral raise:171  squat:238  pull Up:149  leg raises:214


In [10]:
# ==== CELL 10: Augmentation (chỉ train) ====
print('Augmenting TRAIN set only...')
X_train_aug, y_train_aug = augment_train(X_train_raw, y_train_raw)

# Val và Test giữ nguyên, KHÔNG augment
print(f'Train after aug : {X_train_aug.shape}')
print(f'Val  (no aug)   : {X_val_raw.shape}')
print(f'Test (no aug)   : {X_test_raw.shape}')

Augmenting TRAIN set only...
  Augment: 4,806 → 19,224 (4×)
Train after aug : (19224, 30, 99)
Val  (no aug)   : (1247, 30, 99)
Test (no aug)   : (1061, 30, 99)


In [11]:
# ==== CELL 11: Normalize (StandardScaler fit trên train) ====
N, T, F = X_train_aug.shape

scaler = StandardScaler()
scaler.fit(X_train_aug.reshape(N, T*F))

def normalize(X_in, sc):
    n, t, f = X_in.shape
    return sc.transform(X_in.reshape(n, t*f)).reshape(n, t, f).astype(np.float32)

X_train_norm = normalize(X_train_aug, scaler)
X_val_norm   = normalize(X_val_raw,   scaler)
X_test_norm  = normalize(X_test_raw,  scaler)

print('=== NORMALIZE DONE ===')
print(f'Train: mean={X_train_norm.mean():.4f}  std={X_train_norm.std():.4f}')
print(f'Val  : {X_val_norm.shape}')
print(f'Test : {X_test_norm.shape}')

# Lưu scaler
scaler_params = {
    'mean' : scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'n_features_in': int(scaler.n_features_in_)
}
with open(os.path.join(OUTPUT_DIR, 'scaler_params.json'), 'w') as f:
    json.dump(scaler_params, f, indent=2)
print('Saved scaler_params.json')

=== NORMALIZE DONE ===
Train: mean=0.0000  std=1.0000
Val  : (1247, 30, 99)
Test : (1061, 30, 99)
Saved scaler_params.json


In [12]:
# ==== CELL 12: Save .npy files ====
saves = [
    ('X_train.npy', X_train_norm), ('X_val.npy', X_val_norm), ('X_test.npy', X_test_norm),
    ('y_train.npy', y_train_aug),  ('y_val.npy', y_val_raw),  ('y_test.npy', y_test_raw),
]
for fname, arr in saves:
    np.save(os.path.join(OUTPUT_DIR, fname), arr)

label_map_str = {str(k): v for k, v in LABEL_MAP.items()}
with open(os.path.join(OUTPUT_DIR, 'label_map.json'), 'w') as f:
    json.dump(label_map_str, f, indent=2)

# Lưu video split info để reference
with open(os.path.join(OUTPUT_DIR, 'split_info.json'), 'w') as f:
    json.dump({
        'method': 'video-level split (no leakage)',
        'train_pct': '~70%', 'val_pct': '~15%', 'test_pct': '~15%',
        'per_class': split_stats
    }, f, indent=2)

print('=== ALL FILES SAVED ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    p    = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(p) / 1024 / 1024
    print(f'  {fname:<25}  {size:6.1f} MB')

=== ALL FILES SAVED ===
  X_test.npy                   12.0 MB
  X_train.npy                 217.8 MB
  X_val.npy                    14.1 MB
  eda_stats.json                0.0 MB
  label_map.json                0.0 MB
  scaler_params.json            0.1 MB
  split_info.json               0.0 MB
  y_test.npy                    0.0 MB
  y_train.npy                   0.1 MB
  y_val.npy                     0.0 MB


In [13]:
# ==== CELL 13: Verify & Summary ====
X_tr = np.load(os.path.join(OUTPUT_DIR, 'X_train.npy'))
y_tr = np.load(os.path.join(OUTPUT_DIR, 'y_train.npy'))

print('=== VERIFY ===')
print(f'X_train : {X_tr.shape}  dtype={X_tr.dtype}')
print(f'mean={X_tr.mean():.4f}  std={X_tr.std():.4f}  NaN={np.isnan(X_tr).sum()}')

print('\n=== VIDEO-LEVEL SPLIT SUMMARY ===')
print(f'{"Class":<15} {"Total":>5} {"Train vids":>10} {"Val vids":>8} {"Test vids":>9}')
for ex, info in split_stats.items():
    print(f'{ex:<15} {info["total"]:>5} {info["train_vids"]:>10} {info["val_vids"]:>8} {info["test_vids"]:>9}')

print()
print(f'Train windows (after aug): {len(X_tr):,}')
print(f'Val   windows: {len(X_val_norm):,}')
print(f'Test  windows: {len(X_test_norm):,}')
print()
print('>>> KHÔNG còn data leakage — sẵn sàng train! <<<')

=== VERIFY ===
X_train : (19224, 30, 99)  dtype=float32
mean=0.0000  std=1.0000  NaN=0

=== VIDEO-LEVEL SPLIT SUMMARY ===
Class           Total Train vids Val vids Test vids
push-up            56         38        9         9
lateral raise      37         25        6         6
squat              29         19        5         5
pull Up            26         18        4         4
leg raises         21         14        3         4

Train windows (after aug): 19,224
Val   windows: 1,247
Test  windows: 1,061

>>> KHÔNG còn data leakage — sẵn sàng train! <<<
